Basically the other notebook is a suboptimal to run on a computer. [MOST OF THE TRANSFORMER CODE FROM HERE](https://medium.com/data-science/build-your-own-transformer-from-scratch-using-pytorch-84c850470dcb).

Length size of 72 was used because 1. It contains 700+ seq. 2.

> This model is extremely inefficient as ive since learned. I decided to use a encoder-dcoder architecture, since I approached it as a "translation" problem, but after I built it I realsed it only requires new embeddings per residue.

# Loading in the data

In [8]:
import torch
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
import numpy as np
from torch import nn
# import torch.nn.functional as F
# import torch.optim as optim


device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using: {device}")

from data_loader import load_data

X, y, AminoAcids, SecondaryStructures, aa_stoi, aa_itos, aa_encode, aa_decode, ss_stoi, ss_itos, ss_encode, ss_decode, PAD_LABEL, set_length = load_data(set_length=72)
dataset = TensorDataset(X, y)

print(f"Dataset imported, size: {len(dataset)}")

Using: mps
Dataset imported, size: 732


# Splitting the Data & Batching

In [9]:
traindata, valdata, testdata = torch.utils.data.random_split(dataset, [0.80, 0.10, 0.10])

print(f"Train size: {len(traindata)}, Val size: {len(valdata)}, Test size: {len(testdata)}")

train_loader = DataLoader(traindata, batch_size=32, shuffle=True)
val_loader = DataLoader(valdata, batch_size=32, shuffle=False)
test_loader = DataLoader(testdata, batch_size=32, shuffle=False)

Train size: 586, Val size: 73, Test size: 73


# Import Model, Config & Train

In [10]:
from model import Transformer

src_vocab_size = len(AminoAcids) + 1
tgt_vocab_size = len(SecondaryStructures) + 1
d_model = 24
num_heads = 4
num_layers = 1
d_ff = 4*d_model
max_seq_length = set_length
dropout = 0.2

transformer = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout)

transformer.eval()

Transformer(
  (encoder_embedding): Embedding(21, 24)
  (decoder_embedding): Embedding(9, 24)
  (positional_encoding): PositionalEncoding()
  (encoder_layers): ModuleList(
    (0): EncoderLayer(
      (self_attn): MultiHeadAttention(
        (W_q): Linear(in_features=24, out_features=24, bias=True)
        (W_k): Linear(in_features=24, out_features=24, bias=True)
        (W_v): Linear(in_features=24, out_features=24, bias=True)
        (W_o): Linear(in_features=24, out_features=24, bias=True)
      )
      (feed_forward): PositionWiseFeedForward(
        (fc1): Linear(in_features=24, out_features=96, bias=True)
        (fc2): Linear(in_features=96, out_features=24, bias=True)
        (relu): ReLU()
      )
      (norm1): LayerNorm((24,), eps=1e-05, elementwise_affine=True, bias=True)
      (norm2): LayerNorm((24,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.2, inplace=False)
    )
  )
  (decoder_layers): ModuleList(
    (0): DecoderLayer(
      (self_at

In [11]:
from train import train_model


transformer = train_model(
    model=transformer,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    tgt_vocab_size=tgt_vocab_size,
    num_epochs=1000,
    learning_rate=0.0001,
    save_path="./data/weights.pth"
)

Epoch: 1, Train Loss: 2.1985, Val Loss: 2.0804
Epoch: 2, Train Loss: 2.0134, Val Loss: 1.9088
Epoch: 3, Train Loss: 1.8571, Val Loss: 1.7762
Epoch: 4, Train Loss: 1.7450, Val Loss: 1.6731
Epoch: 5, Train Loss: 1.6454, Val Loss: 1.5909
Epoch: 6, Train Loss: 1.5675, Val Loss: 1.5238
Epoch: 7, Train Loss: 1.5118, Val Loss: 1.4670
Epoch: 8, Train Loss: 1.4531, Val Loss: 1.4151
Epoch: 9, Train Loss: 1.4047, Val Loss: 1.3681
Epoch: 10, Train Loss: 1.3586, Val Loss: 1.3213
Epoch: 11, Train Loss: 1.3211, Val Loss: 1.2754
Epoch: 12, Train Loss: 1.2818, Val Loss: 1.2333
Epoch: 13, Train Loss: 1.2397, Val Loss: 1.1936
Epoch: 14, Train Loss: 1.2041, Val Loss: 1.1557
Epoch: 15, Train Loss: 1.1738, Val Loss: 1.1226
Epoch: 16, Train Loss: 1.1444, Val Loss: 1.0931
Epoch: 17, Train Loss: 1.1116, Val Loss: 1.0658
Epoch: 18, Train Loss: 1.0822, Val Loss: 1.0425
Epoch: 19, Train Loss: 1.0610, Val Loss: 1.0227
Epoch: 20, Train Loss: 1.0411, Val Loss: 1.0043
Epoch: 21, Train Loss: 1.0213, Val Loss: 0.9884
E

# Test

In [12]:
from testing import test_model

criterion = nn.CrossEntropyLoss(ignore_index=-100)

test_loss, test_accuracy = test_model(
    model=transformer,
    test_loader=test_loader,
    device=device,
    tgt_vocab_size=tgt_vocab_size,
    criterion=criterion
)

Test Loss: 0.4879
Test Accuracy: 82.17%


In [13]:
model = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout).to(device)
model.load_state_dict(torch.load("./data/weights.pth", weights_only=True))

np.random.seed(42)

for i in range(10):
    random_row = np.random.randint(0, len(testdata))
    src_example = testdata[random_row][0].unsqueeze(0).to(device)
    true_labels = testdata[random_row][1].unsqueeze(0).to(device)

    # Teacher-forced decoder input: start token, then previous true labels.
    start_token = torch.zeros((true_labels.size(0), 1), dtype=torch.long, device=device)
    tgt_input = torch.cat([start_token, true_labels[:, :-1]], dim=1)


    model.eval()
    with torch.no_grad():
        prediction = model(src_example, tgt_input)
        predicted_labels = prediction.argmax(-1)
        accuracy = (predicted_labels == true_labels).float().mean().item()

    predicted_sst8 = ss_decode(predicted_labels.squeeze(0).cpu().tolist())
    true_sst8 = ss_decode(true_labels.squeeze(0).cpu().tolist())

    print(f"Accuracy:  {accuracy:.2%}")
    print(f"Predicted: {predicted_sst8}")
    print(f"True:      {true_sst8}")

Accuracy:  88.89%
Predicted: CCCCCCCCCCCHHHHHHHHHHHHHCCCCCHHHHHHHHHHHCCCHHHHHHHHHHHHHHHHHCCCCCCCCCCCC
True:      CCCCCCCCCCHHHHHHHHHHHHHCSSCCHHHHHHHHHHHCCCHHHHHHHHHHHHHHHHHCCCCCCCCCCCCC
Accuracy:  84.72%
Predicted: CCCCCCCCCCCCHHHHHHHHHHHHHCCCCCHHHHHHHHHHHCCCHHHHHHHHHHHHHHHHHHHHHHHHHHCC
True:      CCCSSSCCCCCHHHHHHHHHHHHHCSSCCHHHHHHHHHHHTCCHHHHHHHHHHHHHHHHHHHHHHHHHHHHC
Accuracy:  97.22%
Predicted: CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCHHHHHHHHHHHHHHHHHHHHHHHHCCCCCCC
True:      CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCHHHHHHHHHHHHHHHHHHHHHHHHCCCCCCCC
Accuracy:  86.11%
Predicted: CCCCCCHHHHHHHHSSCHHHHHHHHHHHHHHHHHHHHHHHTTSSCCTGGGTHHHHHHHHHHHHHHHHHHHCC
True:      CCCCCHHHHHHHHSSSCSSHHHHHHHHHHHHHHHHHHHHHTSSSCCGGGTHHHHHHHHHHHHHHHHHHHTTC
Accuracy:  90.28%
Predicted: CCCHHHHHHHHHHHTSCHHHHHHHHHHHHHHHHHHHHHHHTTSSCCTHHHHHHHHHHHHHHHHHHHHHHHCC
True:      CCHHHHHHHHHHHTSSHHHHHHHHHHHHHHHHHHHHHHHHTSSSCCTTHHHHHHHHHHHHHHHHHHHHHHTC
Accuracy:  81.94%
Predicted: CCCCCCCCCCCCCTTTHHHHHHHHHHHHHHHHCTTTCCCCH